# Data Ingestion

**Project:** Airbnb Data Engineering Assessment  
**Goal:** Download London Airbnb data and load into DuckDB

**Why London?**
- ~80,000 listings (large enough for statistical analysis)
- English-language reviews
- Well-documented dataset
- 100+ neighborhoods for geographic analysis


In [ ]:
import duckdb
import pandas as pd
import os

print("Libraries loaded successfully")
print(f"Working directory: {os.getcwd()}")
print(f"DuckDB version: {duckdb.__version__}")

In [ ]:
import urllib.request
import os

os.makedirs('../data/raw', exist_ok=True)

urls = {
    'listings.csv.gz': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/data/listings.csv.gz',
    'reviews.csv.gz': 'http://data.insideairbnb.com/united-kingdom/england/london/2024-09-06/data/reviews.csv.gz',
}

for filename, url in urls.items():
    filepath = f'../data/raw/{filename}'
    if not os.path.exists(filepath):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"Done ({size_mb:.1f} MB)")
    else:
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        print(f"{filename} already exists ({size_mb:.1f} MB)")

In [ ]:
# Check downloaded files

import os
for f in os.listdir('../data/raw'):
    size_mb = os.path.getsize(f'../data/raw/{f}') / 1024 / 1024
    print(f"  📄 {f}: {size_mb:.1f} MB")

In [ ]:
# Load CSVs into DuckDB

# Connect to a database file (will be created if doesn't exist)
con = duckdb.connect('../data/airbnb.db')
print("Connected to DuckDB database!")

# Load listings.csv into a DuckDB table
con.execute("""
    CREATE OR REPLACE TABLE listings AS 
    SELECT * FROM read_csv_auto('../data/raw/listings.csv')
""")
print("Listings table created!")

# Load reviews.csv into a DuckDB table
con.execute("""
    CREATE OR REPLACE TABLE reviews AS 
    SELECT * FROM read_csv_auto('../data/raw/reviews.csv')
""")
print("Reviews table created!")


### Count listings and reviews

In [ ]:
result = con.execute("SELECT COUNT(*) as total FROM listings").fetchone()
print(f" Total listings: {result[0]:,}")

result = con.execute("SELECT COUNT(*) as total FROM reviews").fetchone()
print(f" Total reviews: {result[0]:,}")

### Show schema for listings

In [ ]:
con.execute("DESCRIBE listings").df()

### Show schema for reviews

In [ ]:
con.execute("DESCRIBE reviews").df()

### Data inspection of listings

In [ ]:
con.execute("SELECT * FROM listings LIMIT 5").df()

### Data inspection of reviews

In [ ]:
con.execute("SELECT * FROM reviews LIMIT 5").df()